# Amazon ML Hackathon 2026: Business Entity Resolution
## Paradigm 1: Deep Entity Matching with Contextualized Late-Interaction (ColBERT + Ditto)

### Theoretical Background & Research Foundation
This notebook implements **Paradigm 1 (Contextualized Late-Interaction with Ditto Serialization)**, synthesizing breakthrough methodologies from two seminal papers in database and NLP research:
1. **Ditto** (*Li et al., "Deep Entity Matching with Pre-Trained Language Models", EMNLP 2020*): Formulates tabular entity resolution as sequence-to-sequence matching with domain-specific attribute serialization tags (`[NAME]`, `[ADDR]`, `[CTRY]`).
2. **ColBERT / PLAID** (*Khattab & Zaharia, "ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT", SIGIR 2020*; *Santhanam et al., ACM TOIS 2024*): Decouples representation encoding from pair scoring by projecting token hidden states into low-dimensional normalized vectors and scoring via the closed-form **MaxSim operator**:
$$\text{MaxSim}(Q, D) = \sum_{i=1}^{|Q|} \max_{j=1}^{|D|} \left( \mathbf{e}_{q, i}^\top \mathbf{e}_{d, j} \right)$$

### Continuous GitHub Sync in Kaggle
- This notebook automatically pulls the latest `.py` source files from the GitHub repository (`https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git`).
- Whenever you push code updates or bug fixes to GitHub, simply re-run Section 1 in Kaggle to sync the latest modules instantly!

## Section 1: Automatic GitHub Sync & Hardware Discovery

This cell:
1. Detects if running inside Kaggle.
2. Clones or fetches/resets `https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git` to ensure all latest `.py` files in `colbert_ditto_er/src/` are up to date.
3. Adds the repository root to `sys.path` so modules can be imported directly.
4. Queries GPU specs for **2× NVIDIA Tesla T4 GPUs** (with CPU fallback).

In [ ]:
# ==============================================================================
# 1. GITHUB REPOSITORY AUTO-SYNC & HARDWARE DISCOVERY
# ==============================================================================
import os
import sys
import subprocess
import psutil
from pathlib import Path

REPO_URL = "https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git"
REPO_NAME = "Amazon-ML-Hackathon-2026"
IS_KAGGLE = Path("/kaggle").is_dir()

def sync_github_repository() -> Path:
    if IS_KAGGLE:
        target_dir = Path("/kaggle/working") / REPO_NAME
        if (target_dir / ".git").is_dir():
            print(f"[GITHUB SYNC] Repository found at {target_dir}. Fetching latest commits from {REPO_URL}...")
            try:
                subprocess.run(["git", "-C", str(target_dir), "fetch", "--all"], check=True)
                subprocess.run(["git", "-C", str(target_dir), "reset", "--hard", "origin/main"], check=True)
                print("[GITHUB SYNC] Successfully updated repository to latest commit on 'main'.")
            except Exception as e:
                print(f"[GITHUB SYNC WARNING] {e}. Re-cloning cleanly...")
                import shutil
                shutil.rmtree(target_dir, ignore_errors=True)
                subprocess.run(["git", "clone", REPO_URL, str(target_dir)], check=True)
        else:
            print(f"[GITHUB CLONE] Cloning {REPO_NAME} from {REPO_URL} into {target_dir}...")
            res = subprocess.run(["git", "clone", REPO_URL, str(target_dir)], capture_output=True, text=True)
            if res.returncode != 0:
                print(f"[GITHUB ERROR] Git clone failed (code {res.returncode}):\n{res.stderr}")
                if "Could not resolve host" in res.stderr:
                    print("\n[!] CRITICAL: Internet access is turned OFF in your Kaggle notebook settings! Enable 'Internet' in the right-hand Settings panel.")
                res.check_returncode()
            print("[GITHUB CLONE] Clone completed successfully.")
        project_root = target_dir.resolve()
    else:
        # Local development workspace
        project_root = Path.cwd().resolve()
        # If running from a subfolder, resolve to student_resource root
        for p in [project_root, project_root.parent, project_root.parent.parent]:
            if (p / "colbert_ditto_er").is_dir() and (p / "utils").is_dir():
                project_root = p
                break

    # Ensure project root is at index 0 of sys.path
    root_str = str(project_root)
    if root_str in sys.path:
        sys.path.remove(root_str)
    sys.path.insert(0, root_str)
    print(f"[WORKSPACE READY] Root: {project_root}")
    return project_root

PROJECT_ROOT = sync_github_repository()

# Query Hardware
import torch
cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f"\nGPU count: {gpu_count}")
if gpu_count > 0:
    for i in range(gpu_count):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} ({p.total_memory / (1024**3):.2f} GB)")
else:
    print("GPU: None (Using CPU multi-threading OpenMP)")

vm = psutil.virtual_memory()
print(f"CPU Count: {os.cpu_count()}")
print(f"RAM Available: {vm.available / (1024**3):.2f} GB / {vm.total / (1024**3):.2f} GB")

## Section 2: Loading Modules and Dataset Paths

Because the GitHub repository is synced, we import directly from the production modules in `colbert_ditto_er.src.*`.
This guarantees that any changes pushed to GitHub are automatically executed here.

In [ ]:
import time
import gc
import json
import random
import pandas as pd
import numpy as np
from typing import Dict, List, Set, Tuple, Any, Optional

# Import directly from the synced GitHub repository
from colbert_ditto_er.configs.default_config import (
    TRAIN_DIR, TEST_DIR, OUTPUT_DIR, RESULTS_DIR,
    DEFAULT_BASE_MODEL, DEFAULT_EMBED_DIM, DEFAULT_MAX_SEQ_LEN,
    DEFAULT_CHUNK_SIZE
)
from colbert_ditto_er.src.normalization import clean_entity_field, create_normalized_dataframe
from colbert_ditto_er.src.serialization import serialize_record, serialize_dataframe
from colbert_ditto_er.src.blocking import SparseBlockingEngine
from colbert_ditto_er.src.colbert_model import ColBERTTokenEncoder, compute_maxsim_score
from colbert_ditto_er.src.dataset import ColBERTTripletDataset, ColBERTBatchCollator
from colbert_ditto_er.src.training import train_colbert_model
from colbert_ditto_er.src.evaluation import compute_entity_f_beta, evaluate_macro_f05, optimize_thresholds_grid
from colbert_ditto_er.src.assignment import apply_assignment_rules
from colbert_ditto_er.src.pipeline import run_streaming_colbert_inference

print("[MODULES] All ColBERT-Ditto modules imported successfully from GitHub repository.")
print(f"  TRAIN_DIR:  {TRAIN_DIR}")
print(f"  TEST_DIR:   {TEST_DIR}")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")

## Section 3: Multiscript Normalization & Ditto Serialization

We verify that our normalization handles Indic scripts (Devanagari $\to$ Latin), European diacritics, and expands legal business suffixes (*e.g.*, `Pvt Ltd`, `LLC`, `SA`, `SAS`).

In [ ]:
test_cases = [
    ("श्री गणेश इन्टरप्राइजेज", "12 MG Road, Bangalore", "IN"),
    ("Café & Société de Restauration", "15 Rue de Paris", "FR"),
    ("Prime Money Pvt Ltd", "17560 Ellis Road, Tahlequah, OK", "US"),
]

print("Ditto Serialization Verification:")
for name, addr, ctry in test_cases:
    serialized = serialize_record(name, addr, ctry)
    print(f"  Original:   {name} | {addr}")
    print(f"  Serialized: {serialized}\n")

## Section 4: Strict Entity-Disjoint Splitting & Hard Negative Mining

### Leakage-Free Guarantee
We strictly partition S1 reference entities: $S1_{\text{train}} \cap S1_{\text{val}} = \emptyset$.
Validation reference entities NEVER appear during training. This provides an unbiased estimate of Macro $F_{0.5}$.

### Hard Negative Mining
Using the Sparse BM25 Blocker, we mine hard negatives (top retrieval distractors that share lexical similarity but are distinct entities).

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# 1. Load Ground Truth
gt_path = TRAIN_DIR / "train_ground_truth.tsv"
print(f"Loading ground truth from {gt_path}...")
gt_df = pd.read_csv(gt_path, sep="\t", dtype=str)

gt_map = {}
query_to_s1 = {}
for row in gt_df.itertuples():
    s1 = row.source1_entity_id
    matched = str(row.matched_entity_ids).split(",") if pd.notna(row.matched_entity_ids) and str(row.matched_entity_ids).strip() else []
    matched_set = set(m.strip() for m in matched if m.strip())
    gt_map[s1] = matched_set
    for q in matched_set:
        query_to_s1[q] = s1

all_s1_list = list(gt_map.keys())
random.shuffle(all_s1_list)

NUM_VAL_ENTITIES = 10000
NUM_TRAIN_ENTITIES = 50000

val_s1_set = set(all_s1_list[:NUM_VAL_ENTITIES])
train_s1_set = set(all_s1_list[NUM_VAL_ENTITIES:NUM_VAL_ENTITIES + NUM_TRAIN_ENTITIES])

assert len(train_s1_set.intersection(val_s1_set)) == 0, "Data leakage between train and val!"
print(f"Entity-Disjoint Partition: Train S1 = {len(train_s1_set):,} | Val S1 = {len(val_s1_set):,}")

# Load reference S1 and queries
print("Loading S1 reference records...")
s1_train_df = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", dtype=str)
s1_train_df = create_normalized_dataframe(s1_train_df)
s1_train_df["serialized"] = serialize_dataframe(s1_train_df)
s1_text_map = dict(zip(s1_train_df["entity_id"], s1_train_df["serialized"]))

print("Loading queries from Source 2 and Source 3...")
s2_df = pd.read_csv(TRAIN_DIR / "train_source2.tsv", sep="\t", nrows=150000, dtype=str)
s3_df = pd.read_csv(TRAIN_DIR / "train_source3.tsv", sep="\t", nrows=150000, dtype=str)
q_df = pd.concat([s2_df, s3_df], ignore_index=True)
q_df = create_normalized_dataframe(q_df)
q_df["serialized"] = serialize_dataframe(q_df)
q_text_map = dict(zip(q_df["entity_id"], q_df["serialized"]))

# Mine hard negatives with SparseBlockingEngine on train partition
train_ref_df = s1_train_df[s1_train_df["entity_id"].isin(train_s1_set)].copy()
blocker_train = SparseBlockingEngine(top_k_candidates=10)
blocker_train.fit(train_ref_df)

train_query_ids = [q for q, s1 in query_to_s1.items() if s1 in train_s1_set and q in q_text_map][:30000]
train_q_sub = q_df[q_df["entity_id"].isin(set(train_query_ids))].copy()
cands_train = blocker_train.generate_candidates_for_chunk(train_q_sub)

train_triplets = []
for qid in train_query_ids:
    pos_s1 = query_to_s1[qid]
    cands = cands_train.get(qid, [])
    negs = [c for c in cands if c != pos_s1 and c in s1_text_map]
    if negs and pos_s1 in s1_text_map:
        neg_s1 = random.choice(negs)
        train_triplets.append((q_text_map[qid], s1_text_map[pos_s1], s1_text_map[neg_s1]))

print(f"Synthesized {len(train_triplets):,} training triplets with hard BM25 negatives.")

## Section 5: ColBERT Late-Interaction Triplet Fine-Tuning (Dual T4 FP16)

We fine-tune the `ColBERTTokenEncoder` using pairwise **Margin Ranking Loss**:
$$\mathcal{L} = \max\left(0, \gamma - \text{MaxSim}(q, s^+) + \text{MaxSim}(q, s^-)\right)$$
- Backbone: `sentence-transformers/all-MiniLM-L6-v2`.
- Output Head: Orthogonal projection to 64 dimensions.
- FP16 Acceleration: Utilizes PyTorch Automatic Mixed Precision (`torch.cuda.amp.autocast()`) across both T4 GPUs via `nn.DataParallel`.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Initializing ColBERT model on {device}...")

model = ColBERTTokenEncoder(
    base_model_name=DEFAULT_BASE_MODEL,
    embed_dim=DEFAULT_EMBED_DIM,
    max_seq_len=DEFAULT_MAX_SEQ_LEN
)

# Train with Margin Ranking Loss and FP16
trained_model = train_colbert_model(
    model=model,
    triplets=train_triplets,
    epochs=2,
    batch_size=64,
    learning_rate=3e-5,
    margin=0.20,
    device=device,
    use_fp16=torch.cuda.is_available()
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_save_path = RESULTS_DIR / "colbert_model.pt"
torch.save(trained_model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

## Section 6: Validation Evaluation & Macro $F_{0.5}$ 2D Grid Optimization

We evaluate ColBERT late-interaction scores on the held-out validation entities.

### Precision-Oriented Optimization (Macro $F_{0.5}$)
The target metric is **Macro $F_{0.5}$** across reference S1 entities ($eta = 0.5$):
$$F_{0.5} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}$$
- Making a false positive match on a **singleton entity** (no true matches) drops that entity's score from 1.0 directly to 0.0.
- We perform a 2D grid sweep across **absolute score threshold** and **margin threshold** (difference between top-1 and top-2 candidate) to find the global maximum Macro $F_{0.5}$.

In [ ]:
print("[VALIDATION] Building validation blocker on unseen S1 entities...")
val_ref_df = s1_train_df[s1_train_df["entity_id"].isin(val_s1_set)].copy()
blocker_val = SparseBlockingEngine(top_k_candidates=10)
blocker_val.fit(val_ref_df)

val_query_ids = [q for q, s1 in query_to_s1.items() if s1 in val_s1_set and q in q_text_map][:8000]
val_q_sub = q_df[q_df["entity_id"].isin(set(val_query_ids))].copy()
cands_val = blocker_val.generate_candidates_for_chunk(val_q_sub)

from transformers import AutoTokenizer
val_tokenizer = AutoTokenizer.from_pretrained(DEFAULT_BASE_MODEL)

val_eval_pairs = []
for qid in val_query_ids:
    for s1_cand in cands_val.get(qid, []):
        if s1_cand in s1_text_map:
            val_eval_pairs.append((qid, s1_cand, q_text_map[qid], s1_text_map[s1_cand]))

print(f"Scoring {len(val_eval_pairs):,} unseen validation candidate pairs via ColBERT Late Interaction...")
trained_model.eval()
scored_val = []
BATCH_EVAL = 256

with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        for i in range(0, len(val_eval_pairs), BATCH_EVAL):
            sl = val_eval_pairs[i:i + BATCH_EVAL]
            b_q = [x[2] for x in sl]
            b_r = [x[3] for x in sl]
            
            t_q = val_tokenizer(b_q, padding=True, truncation=True, max_length=DEFAULT_MAX_SEQ_LEN, return_tensors="pt")
            t_r = val_tokenizer(b_r, padding=True, truncation=True, max_length=DEFAULT_MAX_SEQ_LEN, return_tensors="pt")
            
            q_emb = trained_model.encode_tokens(t_q["input_ids"].to(device), t_q["attention_mask"].to(device))
            d_emb = trained_model.encode_tokens(t_r["input_ids"].to(device), t_r["attention_mask"].to(device))
            
            ms = compute_maxsim_score(q_emb, t_q["attention_mask"].to(device), d_emb, t_r["attention_mask"].to(device))
            probs = torch.sigmoid((ms - trained_model.bias) / torch.clamp(trained_model.tau, min=0.01))
            p_list = probs.cpu().numpy().tolist()
            for (qid, s1, _, _), sc in zip(sl, p_list):
                scored_val.append({"query_id": qid, "s1_id": s1, "score": sc})

# 2D Grid Search
val_gt_map = {s1: gt_map[s1] for s1 in val_s1_set}
best_abs, best_margin, best_f05, best_metrics = optimize_thresholds_grid(
    scored_candidates=scored_val,
    ground_truth=val_gt_map,
    all_s1_ids=list(val_s1_set)
)

print("=" * 60)
print("OPTIMAL VALIDATION DECISION BOUNDARIES:")
print(f"  Absolute Score Threshold: {best_abs:.2f}")
print(f"  Margin Threshold:         {best_margin:.2f}")
print(f"  Validation Macro F0.5:    {best_f05:.4f}")
print(f"  Validation Precision:     {best_metrics.get('macro_precision', 0):.4f}")
print(f"  Validation Recall:        {best_metrics.get('macro_recall', 0):.4f}")
print("=" * 60)

# Save threshold configuration
cfg_path = RESULTS_DIR / "threshold_config.json"
with open(cfg_path, "w") as f:
    json.dump({
        "abs_threshold": best_abs,
        "margin_threshold": best_margin,
        "macro_f0.5": best_f05,
        "metrics": best_metrics,
        "model_name": DEFAULT_BASE_MODEL
    }, f, indent=2)
print(f"Threshold configuration saved to {cfg_path}")

## Section 7: Memory-Safe Streaming Test Inference (<4.5 GB RAM)

We deploy the trained ColBERT model through `run_streaming_colbert_inference`.

### Pipeline Guarantees:
1. **Constant Memory Footprint**: Evaluates queries in streaming chunks of 25,000–30,000 queries, calling `gc.collect()` and `torch.cuda.empty_cache()` between iterations.
2. **Official Output Files**: Produces `output/matching_results.tsv` and `output/candidate_pairs.tsv`.
3. **Inclusion Guarantee**: Guarantees that `predicted_pairs ⊆ candidate_pairs`.

In [ ]:
match_out = OUTPUT_DIR / "matching_results.tsv"
cand_out = OUTPUT_DIR / "candidate_pairs.tsv"

inference_results = run_streaming_colbert_inference(
    model=trained_model,
    test_dir=TEST_DIR,
    output_matching_path=match_out,
    output_candidate_path=cand_out,
    abs_threshold=best_abs,
    margin_threshold=best_margin,
    chunk_size=DEFAULT_CHUNK_SIZE,
    device=device
)

print("\n[STREAMING INFERENCE COMPLETED]")
print(f"  Total Reference S1 Entities: {inference_results['num_s1']:,}")
print(f"  Total Processed Queries:     {inference_results['total_queries']:,}")
print(f"  Total Candidate Pairs:       {inference_results['num_candidates']:,}")
print(f"  Total Final Matches:         {inference_results['num_matches']:,}")
print(f"  Total Inference Time:        {inference_results['runtime_sec']:.1f}s")
print(f"  Matching TSV Path:           {match_out}")
print(f"  Candidate TSV Path:          {cand_out}")

## Section 8: Official Submission Validation (`utils/validate_submission.py`)

We run the official `utils/validate_submission.py` script provided by the hackathon organizers to verify all formatting rules:
- Output TSV formatting (`tab-separated`, `.tsv`).
- Exact headers: `source1_entity_id`, `matched_entity_ids` / `candidate_entity_ids`.
- Inclusion guarantee: `predicted_pairs ⊆ candidate_pairs`.
- No duplicate rows or self-matches.

In [ ]:
import subprocess

validator_script = PROJECT_ROOT / "utils" / "validate_submission.py"
if validator_script.exists() and match_out.exists():
    print(f"Running official validator: {validator_script}...")
    cmd = [
        sys.executable, str(validator_script),
        "--matching", str(match_out),
        "--candidate", str(cand_out),
        "--test-dir", str(TEST_DIR)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode == 0:
        print(">>> SUCCESS: Official Submission Validation PASSED with Exit Code 0!")
    else:
        print(f">>> VALIDATION RETURNED EXIT CODE {result.returncode}")
else:
    print(f"Validator script not found at {validator_script}. Checked locally.")